In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import shuffle
import os

In [2]:
# Configuration
NUM_CLIENTS = 5
OUTPUT_DIR = 'mnist_split_data_new'
RANDOM_STATE = 42

In [3]:
# Load MNIST dataset
print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print(f"Original train shape: {x_train.shape}")
print(f"Original test shape: {x_test.shape}")

Loading MNIST dataset...
Original train shape: (60000, 28, 28)
Original test shape: (10000, 28, 28)


In [4]:
# Normalize data to [0, 1]
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

print("Data normalized to [0, 1]")

Data normalized to [0, 1]


In [5]:
# Combine train and test sets
x_combined = np.concatenate([x_train_norm, x_test_norm], axis=0)
y_combined = np.concatenate([y_train, y_test], axis=0)

print(f"Combined dataset shape: {x_combined.shape}")
print(f"Total samples: {len(x_combined)}")

Combined dataset shape: (70000, 28, 28)
Total samples: 70000


In [6]:
# Shuffle the combined dataset
x_combined, y_combined = shuffle(x_combined, y_combined, random_state=RANDOM_STATE)
print("Dataset shuffled")

Dataset shuffled


In [7]:
# Split data using Stratified K-Fold to maintain class distribution
print(f"\nSplitting data into {NUM_CLIENTS} clients...")
skf = StratifiedKFold(n_splits=NUM_CLIENTS, shuffle=True, random_state=RANDOM_STATE)

parts_x = []
parts_y = []

for fold, (train_idx, test_idx) in enumerate(skf.split(x_combined, y_combined), start=1):
    parts_x.append(x_combined[test_idx])
    parts_y.append(y_combined[test_idx])
    print(f"Client {fold}: {len(test_idx)} samples")


Splitting data into 5 clients...
Client 1: 14000 samples
Client 2: 14000 samples
Client 3: 14000 samples
Client 4: 14000 samples
Client 5: 14000 samples


In [8]:
# Further split each client's data into train and test sets
# Maintain the original 60k/10k ratio (approximately 85.7% train, 14.3% test)
original_train_ratio = 60000 / (60000 + 10000)
test_ratio = 1 - original_train_ratio

print(f"\nSplitting each client's data (train ratio: {original_train_ratio:.4f})...")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, (px, py) in enumerate(zip(parts_x, parts_y), start=1):
    # Split into train and test
    x_tr, x_te, y_tr, y_te = train_test_split(
        px, py, 
        test_size=test_ratio, 
        stratify=py, 
        random_state=RANDOM_STATE
    )
    
    # Save to npz file
    output_file = os.path.join(OUTPUT_DIR, f'mnist_part{i}.npz')
    np.savez(
        output_file,
        x_train=x_tr,
        y_train=y_tr,
        x_test=x_te,
        y_test=y_te
    )
    
    print(f"Client {i}: Train={len(x_tr)}, Test={len(x_te)} -> Saved to {output_file}")


Splitting each client's data (train ratio: 0.8571)...
Client 1: Train=11999, Test=2001 -> Saved to mnist_split_data_new\mnist_part1.npz
Client 2: Train=11999, Test=2001 -> Saved to mnist_split_data_new\mnist_part2.npz
Client 3: Train=11999, Test=2001 -> Saved to mnist_split_data_new\mnist_part3.npz
Client 4: Train=11999, Test=2001 -> Saved to mnist_split_data_new\mnist_part4.npz
Client 5: Train=11999, Test=2001 -> Saved to mnist_split_data_new\mnist_part5.npz


In [9]:
# Verify the saved data
print("\n=== Verification ===")
total_train = 0
total_test = 0

for i in range(1, NUM_CLIENTS + 1):
    data = np.load(os.path.join(OUTPUT_DIR, f'mnist_part{i}.npz'))
    train_size = len(data['x_train'])
    test_size = len(data['x_test'])
    
    total_train += train_size
    total_test += test_size
    
    print(f"Client {i}: Train={train_size}, Test={test_size}")
    
    # Check class distribution
    unique, counts = np.unique(data['y_train'], return_counts=True)
    print(f"  Class distribution: {dict(zip(unique, counts))}")

print(f"\nTotal train samples: {total_train}")
print(f"Total test samples: {total_test}")
print(f"Total samples: {total_train + total_test}")
print("\n✓ Data splitting completed successfully!")


=== Verification ===
Client 1: Train=11999, Test=2001
  Class distribution: {0: 1183, 1: 1350, 2: 1198, 3: 1224, 4: 1170, 5: 1083, 6: 1179, 7: 1250, 8: 1170, 9: 1192}
Client 2: Train=11999, Test=2001
  Class distribution: {0: 1183, 1: 1350, 2: 1198, 3: 1225, 4: 1170, 5: 1083, 6: 1178, 7: 1250, 8: 1170, 9: 1192}
Client 3: Train=11999, Test=2001
  Class distribution: {0: 1184, 1: 1350, 2: 1198, 3: 1224, 4: 1170, 5: 1082, 6: 1178, 7: 1250, 8: 1170, 9: 1193}
Client 4: Train=11999, Test=2001
  Class distribution: {0: 1184, 1: 1351, 2: 1198, 3: 1224, 4: 1170, 5: 1082, 6: 1178, 7: 1249, 8: 1170, 9: 1193}
Client 5: Train=11999, Test=2001
  Class distribution: {0: 1184, 1: 1351, 2: 1198, 3: 1224, 4: 1169, 5: 1082, 6: 1178, 7: 1250, 8: 1170, 9: 1193}

Total train samples: 59995
Total test samples: 10005
Total samples: 70000

✓ Data splitting completed successfully!
